In [ ]:
"""
=============================================================
Phase 3 — Step 2: Download CHIRPS Rainfall + คำนวณ P_eff
Mae Na Rua Sub-District, Phayao (19.05°N, 99.80°E)
Period: 2020–2024
=============================================================
PREREQUISITES:
  pip install requests pandas numpy

CHIRPS: Climate Hazards Group InfraRed Precipitation with Station data
  - Resolution: 0.05° (~5.5 km)
  - Free, no API key required
  - URL: https://data.chc.ucsb.edu/products/CHIRPS-2.0/
=============================================================
"""

import requests
import numpy as np
import pandas as pd
from pathlib import Path
import os

# ── พิกัด ต.แม่นาเรือ ─────────────────────────────────────────
TARGET_LAT = 19.05
TARGET_LON = 99.80

# ── Download CHIRPS daily (ปี 2020–2024) ─────────────────────
# CHIRPS ใช้ GeoTIFF รายวัน → extract ค่าที่พิกัดด้วย rasterio

print("=== Download CHIRPS Rainfall ===")
print("Method: GEE export (แนะนำ) หรือ rasterio extract จาก GeoTIFF")

# ── วิธีที่ 1: ใช้ข้อมูลจาก GEE ที่ export ไว้แล้ว ────────────
# ถ้ามีไฟล์ CHIRPS_daily_maenaerua.csv จาก GEE แล้ว ใช้ทางนี้

CHIRPS_FROM_GEE = 'CHIRPS_daily_maenaerua.csv'

if os.path.exists(CHIRPS_FROM_GEE):
    print(f"✅ พบไฟล์ GEE: {CHIRPS_FROM_GEE}")
    rain = pd.read_csv(CHIRPS_FROM_GEE)
    print(f"Columns: {rain.columns.tolist()}")
    print(rain.head(3))

else:
    print(f"ไม่พบ {CHIRPS_FROM_GEE} — ใช้ CHIRPS API แทน")

    # ── วิธีที่ 2: Download CHIRPS ผ่าน UCSB server ────────────
    print("\nDownloading CHIRPS daily GeoTIFF...")

    try:
        import rasterio
        from rasterio.crs import CRS
        import io

        records = []

        for year in range(2020, 2025):
            print(f"  Year {year}...", end='', flush=True)
            for month in range(1, 13):
                # จำนวนวันในเดือน
                import calendar
                n_days = calendar.monthrange(year, month)[1]

                for day in range(1, n_days+1):
                    date_str = f"{year}.{month:02d}.{day:02d}"
                    url = (
                        f"https://data.chc.ucsb.edu/products/CHIRPS-2.0/"
                        f"global_daily/tifs/p05/{year}/"
                        f"chirps-v2.0.{date_str}.tif.gz"
                    )

                    try:
                        resp = requests.get(url, timeout=30)
                        if resp.status_code == 200:
                            import gzip
                            with gzip.open(io.BytesIO(resp.content)) as gz:
                                data = gz.read()
                            with rasterio.open(io.BytesIO(data)) as src:
                                val = list(src.sample([(TARGET_LON, TARGET_LAT)]))[0][0]
                            val = max(0, float(val)) if val != -9999 else 0.0
                        else:
                            val = np.nan
                    except:
                        val = np.nan

                    records.append({
                        'date': f"{year}-{month:02d}-{day:02d}",
                        'precipitation': val
                    })

            print(" done")

        rain = pd.DataFrame(records)
        rain.to_csv('CHIRPS_daily_raw.csv', index=False)
        print("✅ Saved: CHIRPS_daily_raw.csv")

    except ImportError:
        print("❌ ต้องติดตั้ง rasterio: pip install rasterio")
        raise


# ── แปลง format ให้ตรงกัน ─────────────────────────────────────
if 'date' not in rain.columns:
    # GEE export อาจมีชื่อ column ต่างกัน
    print("\nColumns ที่มี:", rain.columns.tolist())
    # ปรับตาม column จริง
    date_col  = [c for c in rain.columns if 'date' in c.lower()][0]
    rain_col  = [c for c in rain.columns if any(
                 x in c.lower() for x in ['rain','prec','chirps','total'])][0]
    rain = rain.rename(columns={date_col:'date', rain_col:'precipitation'})

rain['date']          = pd.to_datetime(rain['date'])
rain['precipitation'] = pd.to_numeric(rain['precipitation'], errors='coerce').fillna(0)
rain['precipitation'] = rain['precipitation'].clip(lower=0)

print(f"\nRainfall daily stats (mm/day):")
print(rain['precipitation'].describe().round(2))

# ── คำนวณ P_eff (USDA SCS Method) ────────────────────────────
# P_eff = 0.8 × P_week  (สำหรับ paddy field)
# สำหรับ upland crops: P_eff = max(0, P_week - 5) * 0.85

rain['year'] = rain['date'].dt.year
rain['week'] = rain['date'].dt.isocalendar().week.astype(int)

P_weekly = rain.groupby(['year','week']).agg(
    P_mm_week = ('precipitation', 'sum'),
    n_days    = ('precipitation', 'count'),
).reset_index()

# คำนวณ P_eff สองวิธี
P_weekly['P_eff_paddy']  = P_weekly['P_mm_week'] * 0.8   # paddy/rice
P_weekly['P_eff_upland'] = (                               # upland crops
    np.maximum(0, P_weekly['P_mm_week'] - 5) * 0.85
)

print(f"\nWeekly rainfall stats:")
print(P_weekly['P_mm_week'].describe().round(2))

# ── Sanity check ──────────────────────────────────────────────
print("\n=== Sanity Check ===")
annual_rain = P_weekly.groupby('year')['P_mm_week'].sum()
print("ปริมาณฝนรายปี (mm/year) — คาดหวัง 1,000–1,400 mm สำหรับพะเยา:")
print(annual_rain.round(0).to_string())

wet_dry = P_weekly.groupby(
    P_weekly['week'].apply(lambda w:
        'Wet (wk 18–44)' if 18<=w<=44 else 'Dry (wk 1–17, 45–52)')
)['P_mm_week'].mean()
print("\nค่าเฉลี่ยฝนต่อสัปดาห์ แยก season:")
print(wet_dry.round(1).to_string())

# ── Export ────────────────────────────────────────────────────
P_weekly.to_csv('CHIRPS_weekly_phayao_2020_2024.csv', index=False)
print("\n✅ Saved: CHIRPS_weekly_phayao_2020_2024.csv")
print(f"   Columns: {P_weekly.columns.tolist()}")


# ── PART C: รวม ET₀ + Rainfall เป็น climate_weekly ───────────
print("\n=== รวม ET₀ + Rainfall ===")

ET0_file = 'ET0_weekly_phayao_2020_2024.csv'
if os.path.exists(ET0_file):
    ET0 = pd.read_csv(ET0_file)
    climate = ET0.merge(P_weekly, on=['year','week'], how='left')
    climate['P_mm_week'] = climate['P_mm_week'].fillna(0)

    # คำนวณ deficit (ET₀ - P) — ค่าบวก = ต้องการน้ำเพิ่ม
    climate['deficit_mm'] = climate['ET0_mm_week'] - climate['P_eff_paddy']

    # SPI-4 (4-week Standardized Precipitation Index)
    climate = climate.sort_values(['year','week']).reset_index(drop=True)
    climate['P_4week'] = climate['P_mm_week'].rolling(4, min_periods=4).sum()
    from scipy.stats import zscore
    climate['SPI_4'] = climate.groupby('week')['P_4week'].transform(
        lambda x: zscore(x, ddof=1) if len(x)>1 else 0
    )

    climate.to_csv('climate_weekly_phayao_2020_2024.csv', index=False)
    print("✅ Saved: climate_weekly_phayao_2020_2024.csv")
    print(f"   Rows: {len(climate)} | Columns: {climate.columns.tolist()}")

    print("\nSample (week 26–30, 2022 = wet season peak):")
    sample = climate[(climate['year']==2022) & (climate['week'].between(26,30))]
    print(sample[['year','week','ET0_mm_week','P_mm_week',
                  'P_eff_paddy','deficit_mm']].to_string(index=False))
else:
    print(f"⚠️ ไม่พบ {ET0_file} — รัน phase3_step1_era5_download_ET0.py ก่อน")

print("\n📌 NEXT STEP: zone delineation → คำนวณ NIR_A และ GID_B")

=== Download CHIRPS Rainfall ===
Method: GEE export (แนะนำ) หรือ rasterio extract จาก GeoTIFF
ไม่พบ CHIRPS_daily_maenaerua.csv — ใช้ CHIRPS API แทน

  Year 2020... done
  Year 2021... done
  Year 2022... done
  Year 2023... done
  Year 2024... done
✅ Saved: CHIRPS_daily_raw.csv

Rainfall daily stats (mm/day):
count    1827.00
mean        4.14
std         9.67
min         0.00
25%         0.00
50%         0.00
75%         2.60
max        96.64
Name: precipitation, dtype: float64

Weekly rainfall stats:
count    262.00
mean      28.85
std       36.38
min        0.00
25%        0.00
50%       13.99
75%       42.15
max      165.71
Name: P_mm_week, dtype: float64

=== Sanity Check ===
ปริมาณฝนรายปี (mm/year) — คาดหวัง 1,000–1,400 mm สำหรับพะเยา:
year
2020    1472.0
2021    1523.0
2022    1832.0
2023    1110.0
2024    1621.0

ค่าเฉลี่ยฝนต่อสัปดาห์ แยก season:
week
Dry (wk 1–17, 45–52)     7.7
Wet (wk 18–44)          48.7

✅ Saved: CHIRPS_weekly_phayao_2020_2024.csv
   Columns: ['year', 'week